In [1]:
# 1. Clone the model repository to Kaggle's working directory
!git clone --depth 1 https://huggingface.co/intfloat/multilingual-e5-base /kaggle/working/multilingual-e5-base

Cloning into '/kaggle/working/multilingual-e5-base'...
remote: Enumerating objects: 25, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (21/21), done.
remote: Total 25 (delta 2), reused 24 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (25/25), 47.03 KiB | 7.84 MiB/s, done.
Resolving deltas: 100% (2/2), done.
Filtering content: 100% (10/10), 4.95 GiB | 186.52 MiB/s, done.


In [2]:
%%writefile context_aug_test.py
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

EMB_DIR = "/kaggle/input/datasets/pritomda/hallu-rag-dataset-curation-bn/emb-rag-cur-ben"

corpus_embeddings = np.fromfile(f"{EMB_DIR}/master_embeddings.npy", dtype=np.float16).reshape(-1, 768)
corpus_df = pd.read_parquet(f"{EMB_DIR}/master_text.parquet")
assert len(corpus_df) == corpus_embeddings.shape[0], "text/embedding row mismatch — never reorder/filter one without the other"
corpus_texts = corpus_df["text_chunk"].tolist()
corpus_embeddings = torch.tensor(corpus_embeddings, dtype=torch.float16).cuda()

MODEL_NAME = "/kaggle/working/multilingual-e5-base"      # re-clone this before running, see note above
embed_model = SentenceTransformer(MODEL_NAME).cuda().eval()

@torch.no_grad()
def embed_e5(texts, prefix, batch_size=64, show_progress_bar=False):
    prefixed = [f"{prefix}: {str(t)}" for t in texts]
    embs = embed_model.encode(
        prefixed, batch_size=batch_size, normalize_embeddings=True,
        convert_to_tensor=True, show_progress_bar=show_progress_bar,
    )
    return embs.half().cuda()

def retrieve_topk(query_vecs, k=5, batch_size=256):
    all_indices = [] 
    all_values = []
    for i in range(0, query_vecs.size(0), batch_size):
        batch_queries = query_vecs[i : i + batch_size]
        sims = batch_queries @ corpus_embeddings.T
        topk = sims.topk(k, dim=1)
        all_indices.append(topk.indices)
        all_values.append(topk.values)
        del sims
    return torch.cat(all_indices, dim=0), torch.cat(all_values, dim=0)

def build_augmented_context(existing_context, retrieved_texts, retrieved_vecs, context_vec, sim_threshold=0.95, max_chars=1800):
    if context_vec is not None:
        sims_to_existing = retrieved_vecs @ context_vec
        valid_retrieved = [t for t, s in zip(retrieved_texts, sims_to_existing.tolist()) if s < sim_threshold]
    else:
        valid_retrieved = retrieved_texts[:]
    pieces = valid_retrieved[::-1]
    if existing_context:
        pieces.append(existing_context)
    combined_text = "\n".join(pieces)
    if len(combined_text) > max_chars:
        return combined_text[-max_chars:]
    return combined_text

# --- Test set: columns are id, context, prompt_bn, response_bn ---
TEST_PATH = "/kaggle/input/competitions/bengali-hallucination/test set.csv"

df = pd.read_csv(TEST_PATH)

# FIX: Remove literal "[NULL]" strings from the test set before filling NAs,
# so they don't get treated as valid text to append during augmentation.
df["context"] = df["context"].replace("[NULL]", "").fillna("")

query_texts = df["prompt_bn"].tolist()           # prompt_bn = this dataset's question-equivalent
query_vecs = embed_e5(query_texts, prefix="query", show_progress_bar=True)
idxs, _ = retrieve_topk(query_vecs, k=3)          # k=3 to match context_aug_reverse.py / TOP_K
retrieved_vecs_all = corpus_embeddings[idxs]

non_empty_mask = df["context"].str.strip() != ""
context_vecs_all = torch.zeros(len(df), corpus_embeddings.shape[1], dtype=torch.float16, device="cuda")
if non_empty_mask.any():
    context_vecs_all[non_empty_mask.values] = embed_e5(
        df.loc[non_empty_mask, "context"].tolist(), prefix="query"
    )

augmented_contexts = []
for i, row in df.iterrows():
    retrieved_texts = [corpus_texts[j] for j in idxs[i].tolist()]
    ctx_vec = context_vecs_all[i] if non_empty_mask.iloc[i] else None
    augmented_contexts.append(
        build_augmented_context(row["context"], retrieved_texts, retrieved_vecs_all[i], ctx_vec)
    )

df["context_augmented"] = augmented_contexts

# infer_val_probs.py's format_sequence expects 'question'/'answer' — rename
# here rather than touching that script, since it's a pure rename.
df = df.rename(columns={"prompt_bn": "question", "response_bn": "answer"})

df.to_parquet("test_augmented.parquet")
print(df.head(5))
print("done!", len(df), "rows")

Writing context_aug_test.py


In [3]:
!python3 context_aug_test.py

Loading weights: 100%|█| 199/199 [00:00<00:00, 1431.69it/s, Materializing param=
XLMRobertaModel LOAD REPORT from: /kaggle/working/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████████████████████████████| 40/40 [00:04<00:00,  8.87it/s]
   id  ...                                  context_augmented
0   1  ...  Title: News [মতামত - ২৩ আগস্ট ২০১৫, ০০:২৩]: হু...
1   2  ...  Title: Dictionary: ft. (ফুট .) \n Content: "ফু...
2   3  ...  Title: বাগধারা: চরাচর \n Content: বাগধারা "চরা...
3   4  ...  Title: Dictionary: pled (স্বীকার) \n Content: ...
4   5  ...  Title: Dictionary: grouchy (খিট্খিটে) \n Conte...

[5 rows x 5 columns]
done! 2516 rows


In [4]:
!rm -rf /kaggle/working/multilingual-e5-base

In [5]:
import pandas as pd
pd.set_option('display.max_colwidth', None)
tf = pd.read_parquet("/kaggle/working/test_augmented.parquet")
tf.sample(10)

,id,context,question,answer,context_augmented
2481,2482,খামিস মোহাম্মদ আল-মাররি (জন্ম: ৬ জুলাই ১৯৮৪) হলেন একজন কাতারি ফুটবল রেফারি। তিনি ২০১০ সাল থেকে ফিফা এবং এএফসি-র আন্তর্জাতিক রেফারি তালিকায় স্থান পেয়েছেন।\nএছাড়াও তিনি ঘরোয়া ফুটবলে কাতার স্টার্স লিগের ম্যাচ পরিচালনা করে থাকেন।,খামিস আল-মাররি কবে জন্মগ্রহণ করেন?,৬ জুলাই ২০১০,"Title: মুহাম্মাদ আল-খারাশী \n Content: জীবনী প্রাথমিক জীবন এবং শিক্ষা আল-খারশি ১০১০ হিজরীতে (১৬১০ খ্রিস্টাব্দ) জন্মগ্রহণ করেন এবং কায়রোতে বসবাস করেন। তাকে আল-খারাশি (আল-খারাশী নামেও পরিচিত) বলা হত, কারণ তিনি বুহাইরা গভর্নরেটের আবু-খারাশ গ্রাম থেকে এসেছিলেন। আল-খারশি তার পিতা জামাল আল-দীন আবদুল্লাহ বিন আলী আল-খারশিসহ একদল পণ্ডিত ও ব্যক্তিত্ব দ্বারা শিক্ষিত হন, যিনি আল-খারশিকে বিজ্ঞানের প্রতি ভালবাসা এবং জ্ঞানের আকাঙ্ক্ষা জাগিয়ে তুলেছিলেন।\nTitle: মারঈ আল-কারমি \n Content: মারঈ ইবনে আবু বকর আহমদ আল-কারমি (; ১৫৮০–১৬২৪) (সাধারনত মারঈ ইবনে ইউসুফ আল-কারমি নামে পরিচিত নামে পরিচিত ছিলেন) ছিলেন মুসলিম পণ্ডিত এবং সর্বাধিক বিখ্যাত হাম্বলি পণ্ডিতদের অন্যতম। তিনি তুলকার্ম শহরে জন্মগ্রহণ করেন এবং কায়রো শহরে মারা যান। তিনি বিপুল সংখ্যক ইসলামি বইয়ের রচয়িতা। জীবনী মারঈ আল-কারমি ষোড়শ শতাব্দীতে ১৫৮০ সালের এপ্রিল মাসে ফিলিস্তিনের তুলকার্ম শহরে জন্মগ্রহণ করেন। তার জন্মের বছর সম্পর্কে মুসলিম পণ্ডিতদের মধ্যে মতপার্থক্য রয়েছে।\nTitle: মাহমুদ খামিস \n Content: প্রারম্ভিক জীবন মাহমুদ খামিস সাইদ খামিস আল হাম্মাদি ১৯৮৭ সালের ২৮শে অক্টোবর তারিখে সংযুক্ত আরব আমিরাতের আবুধাবিতে জন্মগ্রহণ করেছেন এবং সেখানেই তার শৈশব অতিবাহিত করেছেন। আন্তর্জাতিক ফুটবল ২০০৭ সালের ১৭ই নভেম্বর তারিখে, ২০ বছর ও ২০ দিন বয়সে, বাম পায়ে ফুটবল খেলায় পারদর্শী খামিস বেনিনের বিরুদ্ধে অনুষ্ঠিত প্রীতি ম্যাচে অংশগ্রহণ করার মাধ্যমে আন্তর্জাতিক ফুটবলে সংযুক্ত আরব আমিরাতের হয়ে অভিষেক করেছেন।\nখামিস মোহাম্মদ আল-মাররি (জন্ম: ৬ জুলাই ১৯৮৪) হলেন একজন কাতারি ফুটবল রেফারি। তিনি ২০১০ সাল থেকে ফিফা এবং এএফসি-র আন্তর্জাতিক রেফারি তালিকায় স্থান পেয়েছেন।\nএছাড়াও তিনি ঘরোয়া ফুটবলে কাতার স্টার্স লিগের ম্যাচ পরিচালনা করে থাকেন।"
1005,1006,,নিচের কোন শব্দটি অন্যদের থেকে আলাদা?,Eagle,"Title: Dictionary: different (বিভিন্ন) \n Content: ""বিভিন্ন"" শব্দটির ইংরেজি অনুবাদ বা অর্থ হলো ""different""। এর সমার্থক বাংলা প্রতিশব্দসমূহ হলো: স্বতন্ত্র, বিভিন্ন, ভিন্ন, আলাদা, অন্য, বিসদৃশ, অসদৃশ, পৃথক্, বিলক্ষণ, পর, নানা, নানাপ্রকার, নানারকম, বিশেষ, হরেক, বদ। ব্যবহারিক উদাহরণ বাক্য: Women are different from men, but it is time to say farewell to the politics of difference. This was a variation on the theme which kept the different aspects of money separate.।\nTitle: News [শিক্ষা - ০১ নভেম্বর ২০১৩, ০০:০১]: বাংলা ২য় পত্র \n Content: ‘বড়’—এর বিপরীত শব্দ কোনটি? ক. খাটো খ. ছোট গ. লম্বা ঘ. মাঝারিসঠিক উত্তর: খ. ছোট৪২। ‘বন্ধন’—এর বিপরীত শব্দ কোনটি? ক. বন্দী খ. স্বাধীন গ. খোলামেলা ঘ. মুক্তিসঠিক উত্তর: ঘ. মুক্তি৪৩। ‘বর্ধমান’—এর বিপরীত শব্দ কোনটি? ক. হ্রস্ব খ. ক্ষীয়মাণ গ. চঞ্চল ঘ. ভবিষ্যৎসঠিক উত্তর: খ. ক্ষীয়মাণ৪৪। ‘বিধি’—এর বিপরীত শব্দ কোনটি? ক. প্রথা খ. নিষেধ গ. নিয়ম ঘ. অনিয়মসঠিক উত্তর: খ. নিষেধ৪৫। নিচের কোনটি ‘বিনীত’ শব্দের বিপরীত শব্দ?\nTitle: Dictionary: other (অন্যান্য) \n Content: ""অন্যান্য"" শব্দটির ইংরেজি অনুবাদ বা অর্থ হলো ""other""। এর সমার্থক বাংলা প্রতিশব্দসমূহ হলো: অন্যটি, আর, অন্য বস্তু, অন্যজন, অপর ব্যক্তি, অন্য, অপরজন, অপরটি, আরেকজন, আরেকটি, অন্যান্য, অন্যতর, দ্বিতীয়, অপর, ইতর, ভিন্ন, আড়, অবশিষ্ট, বাকি, অতিরিক্ত, বাড়তি।"
752,753,১৯৮৩ সালে ডেভিড চৌম তার গবেষণা পত্রে ডিজিটাল মুদ্রার ধারণা দিয়েছিলেন।[5] ১৯৯০ সালে তিনি ডিজিক্যাশ (DigiCash) একটি ইলেকট্রনিক নগদ সংস্থা (ক্যাশ কোম্পানি) প্রতিষ্ঠা করেন। আমস্টারডামে তার গবেষণার বাণিজ্যিক বিকাশ ঘটানোর জন্য তিনি এটি করেন।[6],ডিজিটাল মুদ্রার ধারণা প্রথম কে দিয়েছিলেন ?,ডেভিড চৌম,"Title: ডিজিটাল মুদ্রা \n Content: ইলেক্ট্রনিক মুদ্রা অনেক বেসরকারি ব্যাংক বা অন্য আর্থিক প্রতিষ্ঠানে আমানত হিসেবে রাখা যায়। [৪] ডিজিটাল অর্থ কেন্দ্রীয়করণ হতে পারে, যেখানে অর্থ সরবরাহের উপর নিয়ন্ত্রণ কেন্দ্রীয় নিয়ন্ত্রণ থাকে, বা বিকেন্দ্রিত হয়, যার অর্থ সরবরাহের উপর নিয়ন্ত্রণ বিভিন্ন উৎস থেকে হতে পারে। ১৯৮৩ সালে ডেভিড চৌম তার গবেষণা পত্রে ডিজিটাল মুদ্রার ধারণা দিয়েছিলেন। [৫] ১৯৯০ সালে তিনি ডিজিক্যাশ (DigiCash) একটি ইলেকট্রনিক নগদ সংস্থা (ক্যাশ কোম্পা

In [6]:
!pip install unsloth -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 MB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 109.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 57.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 91.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 73.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 87.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/2

In [7]:
%%writefile infer_val_probs.py
import os
import gc
import time
import argparse
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from unsloth import FastLanguageModel

from peft import PeftModel
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore", category=FutureWarning, module="transformers.modeling_attn_mask_utils")
warnings.filterwarnings("ignore", message=".*attention mask API.*")

def format_sequence(row, model_name):
    ctx = f"অনুচ্ছেদ (Context):\n{row['context_augmented']}\n\n" if str(row['context_augmented']).strip() else ""
    q = f"প্রশ্ন (Question):\n{row['question']}\n\n" if pd.notnull(row["question"]) else ""
    a = f"প্রস্তাবিত উত্তর (Proposed Answer):\n{row['answer']}\n\n" if pd.notnull(row["answer"]) else ""
    if "Bangla" in model_name:
        instruction = "Instruction: উপরের অনুচ্ছেদের এবং প্রশ্নের সাথে প্রস্তাবিত উত্তরের সামঞ্জস্য আছে কি?\nFinal Evaluation:"
    else:
        instruction = "Instruction: Determine whether the proposed answer is fully supported by the context.\nFinal Evaluation:"
    return f"{ctx}{q}{a}{instruction}"


class VocabLogitClassifier(nn.Module):
    def __init__(self, prepared_backbone, vocab_size):
        super().__init__()
        self.backbone = prepared_backbone
        self.classification_head = nn.Linear(vocab_size, 1, dtype=torch.float32)

    def get_hidden_and_lmhead(self):
        try:
            inner = self.backbone.base_model.model
            return inner.model, inner.lm_head
        except AttributeError:
            print("Could not resolve `.base_model.model.model` / `.lm_head` on this checkpoint.")
            print("Inspect with: print(self.backbone.base_model.model)")
            print("...and tell me the correct attribute chain for this architecture.")
            raise

    def forward(self, input_ids, attention_mask=None):
        transformer, lm_head = self.get_hidden_and_lmhead()
        hidden_states = transformer(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        last_hidden = hidden_states[:, -1, :]
        vocab_logits = lm_head(last_hidden.to(lm_head.weight.dtype))
        logit = self.classification_head(vocab_logits.float()).squeeze(-1)
        return logit


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--model_name", required=True, help="Tag used only for output filenames, e.g. qwen3_14b")
    parser.add_argument("--base_model_id", required=True, help="HF/unsloth identifier, e.g. unsloth/Qwen3-14B")
    parser.add_argument("--checkpoint_dir", required=True, help="Dir containing the saved LoRA adapter + classification_head.pt")
    parser.add_argument("--val_path", required=True, help="Path to val_augmented.parquet (or test_augmented.parquet)")
    parser.add_argument("--output_dir", required=True)
    parser.add_argument("--max_seq_length", type=int, required=True)
    parser.add_argument("--batch_size", type=int, required=True)
    parser.add_argument("--no_4bit", action="store_true", help="Disable 4-bit loading if a checkpoint needs full precision")
    parser.add_argument("--log_every", type=int, default=10, help="Print a plain-text progress line every N batches")
    args = parser.parse_args()

    rank = int(os.environ.get("LOCAL_RANK", 0))
    world_size = int(os.environ.get("WORLD_SIZE", 1))
    torch.cuda.set_device(rank)
    device = torch.device(f"cuda:{rank}")

    if "14b" in args.base_model_id.lower():
        print(f"[rank {rank}] Note: {args.base_model_id} is the larger model — "
              f"if you hit OOM on a T4 (16GB), drop --batch_size (try 2-4).")

    gc.collect()
    torch.cuda.empty_cache()

    print(f"[rank {rank}] loading base model {args.base_model_id} ...", flush=True)
    base, tokenizer = FastLanguageModel.from_pretrained(
        model_name=args.base_model_id,
        max_seq_length=args.max_seq_length,
        load_in_4bit=not args.no_4bit,
        dtype=None,
        device_map={"": rank},
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"
    tokenizer.truncation_side = "left"

    print(f"[rank {rank}] loading LoRA adapter from {args.checkpoint_dir} ...", flush=True)
    base = PeftModel.from_pretrained(base, args.checkpoint_dir)
    base.eval()

    vocab_size = base.config.vocab_size
    model = VocabLogitClassifier(base, vocab_size)
    head_path = os.path.join(args.checkpoint_dir, "classification_head.pt")
    model.classification_head.load_state_dict(torch.load(head_path, map_location=device))
    model.classification_head.to(device)
    model.eval()

    val_df = pd.read_parquet(args.val_path)
    n = len(val_df)
    bounds = np.linspace(0, n, world_size + 1).astype(int)
    start, end = bounds[rank], bounds[rank + 1]
    shard_df = val_df.iloc[start:end].reset_index(drop=True)
    shard_indices = np.arange(start, end)
    texts = [format_sequence(row, args.base_model_id) for _, row in shard_df.iterrows()]

    n_batches = (len(texts) + args.batch_size - 1) // args.batch_size
    print(f"[rank {rank}] starting inference: {len(texts)} rows, {n_batches} batches", flush=True)

    probs = np.zeros(len(texts), dtype=np.float32)
    t0 = time.time()
    pbar = tqdm(range(0, len(texts), args.batch_size), total=n_batches, desc=f"[rank {rank}]", mininterval=2.0)
    with torch.no_grad():
        for batch_num, i in enumerate(pbar):
            batch_texts = texts[i:i + args.batch_size]
            enc = tokenizer(
                batch_texts, truncation=True, max_length=args.max_seq_length,
                padding=True, return_tensors="pt",
            ).to(device)
            logit = model(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"])
            probs[i:i + len(batch_texts)] = torch.sigmoid(logit).float().cpu().numpy()

            if (batch_num + 1) % args.log_every == 0 or (i + args.batch_size) >= len(texts):
                elapsed = time.time() - t0
                done = min(i + args.batch_size, len(texts))
                rate = done / elapsed if elapsed > 0 else 0.0
                print(f"[rank {rank}] {done}/{len(texts)} rows "
                      f"({batch_num + 1}/{n_batches} batches) — {rate:.1f} rows/s, {elapsed:.0f}s elapsed", flush=True)

    os.makedirs(args.output_dir, exist_ok=True)
    np.save(os.path.join(args.output_dir, f"{args.model_name}_val_probs_rank{rank}.npy"), probs)
    np.save(os.path.join(args.output_dir, f"{args.model_name}_val_indices_rank{rank}.npy"), shard_indices)
    print(f"[rank {rank}] saved {len(probs)} probs (rows {start}:{end}) to {args.output_dir}")


if __name__ == "__main__":
    main()

Writing infer_val_probs.py


In [8]:
!torchrun --nproc_per_node=2 infer_val_probs.py \
    --model_name qwen3-14b \
    --base_model_id unsloth/Qwen3-14B-Base-unsloth-bnb-4bit \
    --checkpoint_dir /kaggle/input/models/pritomda/qwen14b-ft-unsloth-seq1024-r32-ep1-75/pytorch/default/1 \
    --val_path /kaggle/working/test_augmented.parquet \
    --output_dir /kaggle/working/test_probs \
    --max_seq_length 1024 \
    --batch_size 2

W0717 20:23:00.457000 166 torch/distributed/run.py:852] 
W0717 20:23:00.457000 166 torch/distributed/run.py:852] *****************************************
W0717 20:23:00.457000 166 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0717 20:23:00.457000 166 torch/distributed/run.py:852] *****************************************
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
🦥 Unsloth Zoo will now patch everything to make training faster!
[rank 0] Note: unsloth/Qwen3-14B-Base-unsloth-bnb-4bit is the larger model — if you hit OOM on a T4 (16GB), drop --batch_size (try 2-4).
[rank 0] loading base model unsloth/Qwen3-14B-Base-unsloth-bnb-4bit 

In [9]:
!torchrun --nproc_per_node=2 infer_val_probs.py \
    --model_name banglallama-3.1-8b \
    --base_model_id BanglaLLM/BanglaLLama-3.1-8b-bangla-alpaca-orca-instruct-v0.0.1 \
    --checkpoint_dir /kaggle/input/models/sohayswarajroy/banglallama-3-1-8b-bangla-alpaca-orca-ft-unsloth/pytorch/default/1/bengali-hallucination-model-banglallama-8b/final \
    --val_path /kaggle/working/test_augmented.parquet \
    --output_dir /kaggle/working/test_probs \
    --max_seq_length 768 \
    --batch_size 4

W0717 21:03:04.250000 482 torch/distributed/run.py:852] 
W0717 21:03:04.250000 482 torch/distributed/run.py:852] *****************************************
W0717 21:03:04.250000 482 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0717 21:03:04.250000 482 torch/distributed/run.py:852] *****************************************
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
🦥 Unsloth Zoo will now patch everything to make training faster!
[rank 0] loading base model BanglaLLM/BanglaLLama-3.1-8b-bangla-alpaca-orca-instruct-v0.0.1 ...
[rank 1] loading base model BanglaLLM/BanglaLLama-3.1-8b-bangla-alpaca-orca-instruct-v0.0.1 ...
==((====))== 

In [10]:
%%writefile ensemble_predict.py
import os
import argparse
import numpy as np
import pandas as pd

# --- Fitted on the 814-row val set; wiki was tested and dropped ---
# 3-model LR stack (5-fold OOF macro-F1): 0.9220
# 2-model LR stack, qwen3-14b + banglallama-3.1-8b (5-fold OOF macro-F1): 0.9251
ENSEMBLE_MODELS = ["qwen3-14b", "banglallama-3.1-8b"]
LR_COEF = {"qwen3-14b": 3.85, "banglallama-3.1-8b": 3.404}
LR_INTERCEPT = -4.344
THRESHOLD = 0.45

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--probs_dir", required=True,
                         help="Dir containing <model_name>_val_probs_rank{r}.npy")
    parser.add_argument("--n_shards", type=int, default=2,
                         help="Must match --nproc_per_node used during test inference")
    parser.add_argument("--test_path", required=True,
                         help="Path to the test parquet/csv")
    parser.add_argument("--id_col", required=True)
    parser.add_argument("--output_csv", required=True)
    args = parser.parse_args()

    # 1. Read the dataframe FIRST to dynamically get the row count
    if args.test_path.endswith(".parquet"):
        test_df = pd.read_parquet(args.test_path)
    else:
        test_df = pd.read_csv(args.test_path)
        
    n_rows = len(test_df)
    print(f"Loaded {n_rows} rows from {args.test_path}")

    model_probs = {}
    for name in ENSEMBLE_MODELS:
        probs = np.zeros(n_rows, dtype=np.float32)
        for r in range(args.n_shards):
            prob_file = os.path.join(args.probs_dir, f"{name}_val_probs_rank{r}.npy")
            idx_file = os.path.join(args.probs_dir, f"{name}_val_indices_rank{r}.npy")
            
            # 2. Safety check: ensure inference actually finished successfully
            if not os.path.exists(prob_file):
                raise FileNotFoundError(f"Missing shard file: {prob_file}. Did inference crash?")
                
            p = np.load(prob_file)
            idx = np.load(idx_file)
            probs[idx] = p
        model_probs[name] = probs

    combined_logit = LR_INTERCEPT + sum(LR_COEF[m] * model_probs[m] for m in ENSEMBLE_MODELS)
    combined_prob = sigmoid(combined_logit)
    label_pred = (combined_prob > THRESHOLD).astype(int)

    submission = pd.DataFrame({args.id_col: test_df[args.id_col], "label": label_pred})
    submission.to_csv(args.output_csv, index=False)
    
    print(f"Saved {len(submission)} predictions to {args.output_csv}")
    print("Predicted label distribution:\n", submission["label"].value_counts(normalize=True))

if __name__ == "__main__":
    main()

Writing ensemble_predict.py


In [11]:
import pandas as pd
df = pd.read_csv("/kaggle/input/competitions/bengali-hallucination/test set.csv")
df_aug = pd.read_parquet("/kaggle/working/test_augmented.parquet")
print(df.shape)
print(df_aug.shape)

(2516, 4)
(2516, 5)


In [12]:
!python3 ensemble_predict.py \
    --probs_dir /kaggle/working/test_probs \
    --test_path /kaggle/working/test_augmented.parquet \
    --id_col id \
    --output_csv /kaggle/working/submission.csv

Loaded 2516 rows from /kaggle/working/test_augmented.parquet
Saved 2516 predictions to /kaggle/working/submission.csv
Predicted label distribution:
 label
1    0.515898
0    0.484102
Name: proportion, dtype: float64


In [13]:
import pandas as pd

sub = pd.read_csv("/kaggle/working/submission.csv")
test_df = pd.read_parquet("/kaggle/working/test_augmented.parquet")

merged = test_df.merge(sub, on="id", how="inner")
assert len(merged) == len(sub) == len(test_df), "row count mismatch after merge — check id overlap"

sample = merged.sample(n=50, random_state=42)

for _, row in sample.iterrows():
    print(f"id: {row['id']}")
    print(f"context_augmented:\n{row['context_augmented']}")
    print(f"question: {row['question']}")
    print(f"answer: {row['answer']}")
    print(f"label: {row['label']}  ({'faithful' if row['label'] == 1 else 'hallucinated'})")
    print("-" * 80)

id: 618
context_augmented:
ব্যবহারকারীর সংখ্যা অনুসারে বাংলা বিশ্বের সপ্তম বৃহত্তম ভাষা। [১০][১১] বাংলা সার্বভৌম ভাষাভিত্তিক জাতিরাষ্ট্র বাংলাদেশের একমাত্র রাষ্ট্রভাষা তথা সরকারি ভাষা।
Title: বাংলা ভাষা 
 Content: বাংলা ভাষা (বাঙলা, বাঙ্গলা তথা বাঙ্গালা নামেও পরিচিত) একটি ধ্রুপদি ইন্দো-আর্য ভাষা, যা দক্ষিণ এশিয়ার বাঙালি জাতির প্রধান কথ্য ও লেখ্য ভাষা। ২০২৫ সালের হিসাব অনুযায়ী, প্রায় ২৪.২ কোটি মাতৃভাষী এবং আরও প্রায় ৪.৩ কোটি দ্বিতীয় ভাষাভাষীর[৪] সমন্বয়ে বাংলা ভাষা মাতৃভাষীর সংখ্যায় বিশ্বে ষষ্ঠ[৫] এবং মোট ব্যবহারকারীর সংখ্যা অনুসারে বাংলা বিশ্বের সপ্তম বৃহত্তম ভাষা। [৬][৭] বাংলা সার্বভৌম ভাষাভিত্তিক জাতিরাষ্ট্র বাংলাদেশের একমাত্র রাষ্ট্রভাষা তথা সরকারি ভাষা।
Title: ভাষা 
 Content: সে ভাষাও আবার বিভিন্ন সময়ে বিভিন্নভাবে উচ্চারিত হয়ে এসেছে। ফলে এ শতকে মানুষ তার দৈনন্দিন জীবনে যে ভাষা ব্যবহার করে, হাজার বছর আগেকার মানুষের ভাষা ঠিক এমনটি ছিল না। বর্তমান পৃথিবীতে ৭,০৯৯ টি ভাষা প্রচলিত আছে। (তথ্য সূত্রঃ ইথনোলগ ২০তম সংস্করণ)। তার মধ্যে বাংলা একটি ভাষা। ভাষাভাষী জনসংখ্যার দিক দিয়ে বাংল

In [14]:
%%writefile math_verifier_qwen_v5.py
"""Math-specific hallucination check -- v5.

Changes from v4.1:
  1. Batch processing across rows (--batch_size), replacing one-row-at-a-time generation.
  2. Deterministic day-of-week handling: parses weekday names from question/answer,
     asks the tool for a raw day-count (not pre-reduced mod 7), and resolves the
     comparison in code rather than trusting the model's modular arithmetic.
  3. parse_given_answer now handles bare "√N" notation and profit/loss sign words
     (previously "50% loss" and "50% profit" parsed identically).
  4. Logs the actual tool-call expression + parsed given_numeric to the override
     log, not just the final computed value -- needed to debug wrong computations.
  5. Strict single-digit regex for verdict parsing instead of loose substring checks.
"""
try:
    from unsloth import FastLanguageModel
except ImportError:
    raise ImportError("Unsloth is required for FastLanguageModel execution. Please install unsloth before running.")

import os
import re
import json
import math
import fractions
import argparse
import pandas as pd
import torch
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="transformers")

# ---------------------------------------------------------------------------
# COMMAND-LINE ARGUMENTS
# ---------------------------------------------------------------------------
parser = argparse.ArgumentParser(description="Math-specific hallucination check using Qwen tool calling (batched).")
parser.add_argument("--test_csv_path", type=str, default="/kaggle/input/competitions/bengali-hallucination/test set.csv")
parser.add_argument("--submission_path", type=str, default="/kaggle/working/submission.csv")
parser.add_argument("--override_log_path", type=str, default="/kaggle/working/math_overrides.csv")
parser.add_argument("--model_name", type=str, default="unsloth/Qwen3-14B-Instruct")
parser.add_argument("--max_seq_length", type=int, default=2048)
parser.add_argument("--toolcall_max_new_tokens", type=int, default=768)
parser.add_argument("--verdict_max_new_tokens", type=int, default=32)
parser.add_argument("--hallucinated_label_value", type=int, default=0)
parser.add_argument("--batch_size", type=int, default=8,
                     help="Rows processed per generate() call. Larger = faster but more VRAM; "
                          "with a 4-bit 14B model + 768 tool-call tokens, start around 8 on a T4 and adjust.")
parser.add_argument("--checkpoint_every", type=int, default=5,
                     help="Save submission every N BATCHES (not rows, since v5 batches).")
args = parser.parse_args()

CONFIG = {
    "TEST_CSV_PATH": args.test_csv_path,
    "SUBMISSION_PATH": args.submission_path,
    "OVERRIDE_LOG_PATH": args.override_log_path,
    "MODEL_NAME": args.model_name,
    "VERIFIER_MAX_SEQ_LEN": args.max_seq_length,
    "TOOLCALL_MAX_NEW_TOKENS": args.toolcall_max_new_tokens,
    "VERDICT_MAX_NEW_TOKENS": args.verdict_max_new_tokens,
    "HALLUCINATED_LABEL_VALUE": args.hallucinated_label_value,
    "BATCH_SIZE": args.batch_size,
    "CHECKPOINT_EVERY": args.checkpoint_every,
}

# ---------------------------------------------------------------------------
# STEP 1 -- math-question detector (unchanged from v4.1)
# ---------------------------------------------------------------------------
STRONG_MATH_PATTERN = re.compile(
    r"যোগফল|বিয়োগফল|গুণফল|ভাগফল|লসাগু|গসাগু|বর্গমূল|ঘনফল|অনুপাত|"
    r"সমীকরণ|সমকোণ|ত্রিভুজ|বৃত্তের ব্যাসার্ধ|রম্বস|"
    r"\bsin\(|\bcos\(|\btan\(|"
    r"[0-9০-৯]\s*[+×÷=]\s*[0-9০-৯]|[0-9০-৯]+\s*-\s*[0-9০-৯]+|"
    r"ক্ষেত্রফল|পরিসীমা|পরিধি|পূর্ণবর্গ|মৌলিক সংখ্যা|সম্ভাবনা|"
    r"গতিবেগ|ত্বরণ|ভরবেগ|চক্রবৃদ্ধি|পূর্ণসংখ্যা|"
    r"বর্গসংখ্যা|গুণিতক|অঙ্কবিশিষ্ট|দৈবচয়ন|মোলারিটি|ঘনমাত্রা|"
    r"সমবাহু|সমদ্বিবাহু|সদৃশ|সরলরেখা|লম্ব|মধ্যক|"
    r"গড়|গসাগু|লসাগু|ভগ্নাংশ|দশমিক|শতকরা"
)
WEAK_MATH_KEYWORDS = re.compile(
    r"শতাংশ|গড়|টাকা|টাকায়|লাভ|ক্ষতি|সুদ|দিনে|গতিবেগ|দূরত্ব|কিমি|মিনিট|"
    r"প্রার্থী|নির্বাচন|প্যানেল|বৃদ্ধি|ছাড়|বেড়ে|কমে|বেলা|বার|"
    r"ক্ষেত্রফল|পরিসীমা|মোল|ডিগ্রি|কোণ|রূপান্তর|হেক্সাডেসিমেল|"
    r"যোগফল|বিয়োগফল|গুণফল|ভাগফল|মোট|বাকি|প্রতি|জনপ্রতি|মাথাপিছু|"
    r"সেকেন্ড|ঘণ্টা|মিটার|সেন্টিমিটার|গ্রাম|কেজি|লিটার|"
    r"ছাত্র|ছাত্রী|শ্রমিক|সদস্য|আসন|ভোট|"
    r"উত্তীর্ণ|অনুত্তীর্ণ|পাস|ফেল|নম্বর|সংখ্যা|পরিমাণ|মূল্য|দাম|"
    r"আয়|ব্যয়|জমা|খরচ|উপার্জন|বেতন|মাইনে|"
    r"পূর্ববর্তী|বর্তমান|আগে|পরে|"
    r"অর্ধেক|দ্বিগুণ|তিনগুণ|আধা|দেড়|সাড়ে|"
    r"বর্গ|গুণিতক|অঙ্কবিশিষ্ট|দৈর্ঘ্য|প্রস্থ|উচ্চতা|ওজন|বয়স"
)
HAS_DIGIT = re.compile(r"[0-9০-৯]")

def is_math_question(question_text: str) -> bool:
    text = str(question_text)
    if STRONG_MATH_PATTERN.search(text):
        return True
    if not HAS_DIGIT.search(text):
        return False
    factual_pattern = re.compile(
        r"(কত সালে|কবে |জন্মগ্রহণ|মৃত্যুবরণ|প্রতিষ্ঠিত|স্থাপিত|"
        r"কত তারিখে|কোন সালে|কত খ্রি|কত শতকে|"
        r"ধারা অনুযায়ী|আইন|কার স্বাক্ষর|কে ছিলেন|কোথায়|"
        r"কী ছিল|বিজয়ী|বিপুলভাবে|political party|রাজনৈতিক দল|বার্ষিক আয়|"
        r"মতে|অনুযায়ী|অধিষ্ঠিত|নির্বাচনে|ক্ষমতায়)"
    )
    if factual_pattern.search(text):
        return False
    return bool(WEAK_MATH_KEYWORDS.search(text))

# ---------------------------------------------------------------------------
# STEP 2 -- safe execution sandbox & robust parsing
# ---------------------------------------------------------------------------
try:
    import sympy as _real_sympy
    sympy = _real_sympy
except ImportError:
    class _SympyShim:
        Rational = fractions.Fraction
        sqrt = staticmethod(math.sqrt)
        pi = math.pi
        sin = staticmethod(math.sin)
        cos = staticmethod(math.cos)
        tan = staticmethod(math.tan)
        simplify = staticmethod(lambda x: x)
        N = staticmethod(lambda x, n=15: float(x) if not isinstance(x, (int, float)) else x)
    sympy = _SympyShim()

SAFE_BUILTINS = {
    "abs": abs, "round": round, "min": min, "max": max, "sum": sum,
    "int": int, "float": float, "len": len, "pow": pow, "sorted": sorted,
    "range": range, "list": list, "tuple": tuple, "bool": bool, "str": str,
}
MATH_NAMES = {k: v for k, v in math.__dict__.items() if not k.startswith("_")}
SAFE_GLOBALS = {"__builtins__": SAFE_BUILTINS}
SAFE_LOCALS_BASE = {"math": math, "sympy": sympy, "Fraction": fractions.Fraction, **MATH_NAMES}

def run_expression(expr: str):
    local_ns = dict(SAFE_LOCALS_BASE)
    try:
        result = eval(expr, SAFE_GLOBALS, local_ns)
        if hasattr(result, "evalf"):
            result = float(result.evalf())
        elif hasattr(result, "numerator"):
            result = float(result)
        return float(result)
    except Exception:
        return None

BN_DIGITS = str.maketrans("০১২৩৪৫৬৭৮৯", "0123456789")

def parse_given_answer(text: str):
    t = str(text).translate(BN_DIGITS).strip().replace(",", "")
    is_loss = bool(re.search(r"ক্ষতি", t))
    t = t.replace("%", "")

    value = None
    ratio_match = re.search(r"(\d+)\s*:\s*(\d+)", t)
    sqrt_frac_match = re.search(r"(\d+)\s*/\s*√\s*(\d+)", t)
    bare_sqrt_match = re.search(r"√\s*(\d+(?:\.\d+)?)", t)
    frac_match = re.search(r"(-?\d+(?:\.\d+)?)\s*/\s*(-?\d+(?:\.\d+)?)", t)
    pct_match = re.search(r"(-?\d+(?:\.\d+)?)\s*%", text.translate(BN_DIGITS))  # check original (has % still)
    num_match = re.search(r"-?\d+(?:\.\d+)?", t)

    if ratio_match:
        n1, n2 = float(ratio_match.group(1)), float(ratio_match.group(2))
        value = n1 / n2 if n2 != 0 else None
    elif sqrt_frac_match:
        num, den_sqrt = float(sqrt_frac_match.group(1)), float(sqrt_frac_match.group(2))
        value = num / math.sqrt(den_sqrt)
    elif bare_sqrt_match:
        value = math.sqrt(float(bare_sqrt_match.group(1)))
    elif frac_match:
        num, den = float(frac_match.group(1)), float(frac_match.group(2))
        value = num / den if den != 0 else None
    elif pct_match:
        value = float(pct_match.group(1))
    elif num_match:
        value = float(num_match.group())

    if value is not None and is_loss:
        value = -abs(value)
    return value

def _base_compare(v1: float, v2: float) -> bool:
    tolerance = max(0.01, abs(v1) * 0.001) if abs(v1) > 0 else 0.01
    return abs(v1 - v2) < tolerance

def check_mathematical_alignment(computed: float, given_numeric: float) -> bool:
    if _base_compare(computed, given_numeric):
        return True
    if _base_compare(computed * 100, given_numeric) or _base_compare(computed / 100, given_numeric):
        return True
    if abs(computed) > 1e-6 and _base_compare(1 / computed, given_numeric):
        return True
    return False

# --- Day-of-week: resolved deterministically in code, not via LLM verdict ---
BN_WEEKDAYS = {
    "রবিবার": 0, "সোমবার": 1, "মঙ্গলবার": 2, "বুধবার": 3,
    "বৃহস্পতিবার": 4, "শুক্রবার": 5, "শনিবার": 6,
}
BN_WEEKDAY_RE = re.compile("|".join(BN_WEEKDAYS.keys()))

def find_weekday_index(text):
    m = BN_WEEKDAY_RE.search(str(text))
    return BN_WEEKDAYS[m.group(0)] if m else None

# ---------------------------------------------------------------------------
# STEP 3 -- Model setup
# ---------------------------------------------------------------------------
print(f"Loading {CONFIG['MODEL_NAME']} on single GPU for math verification...")
qwen_model, tokenizer_qwen = FastLanguageModel.from_pretrained(
    model_name=CONFIG["MODEL_NAME"],
    max_seq_length=CONFIG["VERIFIER_MAX_SEQ_LEN"],
    load_in_4bit=True,
    dtype=None,
    device_map={"": 0},
)
FastLanguageModel.for_inference(qwen_model)
if tokenizer_qwen.pad_token is None:
    tokenizer_qwen.pad_token = tokenizer_qwen.eos_token
tokenizer_qwen.padding_side = "left"   # required for correct batched generation

TOOLS = [{
    "type": "function",
    "function": {
        "name": "calculate",
        "description": "Evaluate a math expression and return the numeric result.",
        "parameters": {
            "type": "object",
            "properties": {"expression": {"type": "string", "description": "A single evaluable expression."}},
            "required": ["expression"],
        },
    },
}]
TOOL_CALL_RE = re.compile(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", re.DOTALL)
FALLBACK_JSON_RE = re.compile(r"```json\s*(\{.*?\})\s*```", re.DOTALL)
VERDICT_DIGIT_RE = re.compile(r"\b([012])\b")

@torch.no_grad()
def batch_generate(messages_list, max_new_tokens, tools=None):
    prompts = [
        tokenizer_qwen.apply_chat_template(
            msgs, tools=tools, tokenize=False, add_generation_prompt=True, enable_thinking=False,
        )
        for msgs in messages_list
    ]
    enc = tokenizer_qwen(
        prompts, return_tensors="pt", padding=True, truncation=True,
        max_length=CONFIG["VERIFIER_MAX_SEQ_LEN"],
    ).to("cuda:0")
    stop_strs = ["</tool_call>"] if tools else None
    out = qwen_model.generate(
        **enc, max_new_tokens=max_new_tokens,
        do_sample=True, temperature=0.5, top_p=0.95, top_k=20,
        stop_strings=stop_strs,
        tokenizer=tokenizer_qwen if stop_strs else None,
        pad_token_id=tokenizer_qwen.pad_token_id,
    )
    input_len = enc["input_ids"].shape[1]
    return [tokenizer_qwen.decode(out[i][input_len:], skip_special_tokens=True) for i in range(len(messages_list))]


def build_turn1_messages(question):
    return [{
        "role": "user",
        "content": (
            "You are an expert mathematical assistant. Evaluate this Bengali math problem.\n"
            "Instructions:\n"
            "1. Briefly state the formula or logic in 1 short sentence.\n"
            "2. Immediately use the 'calculate' tool to get the numerical result.\n"
            "3. If the question asks which day of the week a date falls on, use the tool "
            "to return ONLY the total number of elapsed days as a plain integer -- "
            "do NOT reduce it modulo 7 yourself, that's handled separately.\n"
            "4. If the input requires no calculation, reply exactly with 'SKIP'.\n\n"
            f"Question: {question}"
        )
    }]


def build_verdict_prompt(question, given_answer_text, computed, given_str):
    return (
        f"You are a mathematical checking judge. Assess the factual alignment of the proposed answer string.\n\n"
        f"Original Question: {question}\n"
        f"Proposed Answer to Verify: {given_answer_text}\n"
        f"Python Calculated True Baseline Value: {computed} (Extracted Comparison Float: {given_str})\n\n"
        f"Task:\n"
        f"If the proposed text is fundamentally correct, equivalent, or expresses the exact value, output 1.\n"
        f"If the proposed text contains a hallucination, calculation error, or wrong numbers, output 0.\n"
        f"If the input string is not a math problem, output 2.\n"
        f"Output EXACTLY a single digit (0, 1, or 2) with absolutely zero trailing tokens.\n"
        f"Verdict:"
    )


def solve_and_verify_batch(questions, answers):
    """Returns list of (is_faithful, computed, expr, given_numeric) per row, same order as inputs."""
    n = len(questions)
    messages_list = [build_turn1_messages(q) for q in questions]
    turn1_outputs = batch_generate(messages_list, CONFIG["TOOLCALL_MAX_NEW_TOKENS"], tools=TOOLS)

    results = [(None, None)] * n
    exprs = [None] * n
    computeds = [None] * n
    given_numerics = [None] * n
    pending_idx, pending_messages = [], []

    for i in range(n):
        turn1 = turn1_outputs[i]
        if "SKIP" in turn1:
            continue
        tool_match = TOOL_CALL_RE.search(turn1) or FALLBACK_JSON_RE.search(turn1)
        if tool_match is None:
            continue
        try:
            call = json.loads(tool_match.group(1))
            if "arguments" in call:
                expr = call["arguments"]["expression"]
            elif "expression" in call:
                expr = call["expression"]
            else:
                expr = list(call.values())[0]
        except Exception:
            continue

        computed = run_expression(expr)
        if computed is None:
            continue
        exprs[i] = expr
        computeds[i] = computed

        # --- Deterministic day-of-week path: bypass the LLM verdict entirely ---
        ref_weekday = find_weekday_index(questions[i])
        ans_weekday = find_weekday_index(answers[i])
        if ref_weekday is not None and ans_weekday is not None:
            predicted_idx = (ref_weekday + int(round(computed))) % 7
            results[i] = (predicted_idx == ans_weekday, computed)
            continue

        given_numeric = parse_given_answer(answers[i])
        given_numerics[i] = given_numeric

        if given_numeric is not None and check_mathematical_alignment(computed, given_numeric):
            results[i] = (True, computed)
            continue

        given_str = str(given_numeric) if given_numeric is not None else answers[i]
        msgs = list(messages_list[i])
        msgs.append({"role": "assistant", "content": turn1})
        msgs.append({"role": "tool", "content": f"calculate({expr}) = {computed}"})
        msgs.append({"role": "user", "content": build_verdict_prompt(questions[i], answers[i], computed, given_str)})
        pending_idx.append(i)
        pending_messages.append(msgs)

    if pending_messages:
        verdict_outputs = batch_generate(pending_messages, CONFIG["VERDICT_MAX_NEW_TOKENS"], tools=None)
        for j, i in enumerate(pending_idx):
            digit_match = VERDICT_DIGIT_RE.search(verdict_outputs[j].strip())
            digit = digit_match.group(1) if digit_match else None
            if digit == "2":
                results[i] = (None, computeds[i])
            elif digit == "1":
                results[i] = (True, computeds[i])
            elif digit == "0":
                results[i] = (False, computeds[i])
            elif given_numerics[i] is not None:
                results[i] = (check_mathematical_alignment(computeds[i], given_numerics[i]), computeds[i])
            # else: leave as (None, computed) -- ambiguous, no override

    return [(results[i][0], results[i][1], exprs[i], given_numerics[i]) for i in range(n)]

# ---------------------------------------------------------------------------
# STEP 4 -- Batched processing loop
# ---------------------------------------------------------------------------
test_df = pd.read_csv(CONFIG["TEST_CSV_PATH"])
test_df = test_df.rename(columns={"prompt_bn": "question", "response_bn": "answer"})
submission = pd.read_csv(CONFIG["SUBMISSION_PATH"])

math_mask = test_df["question"].apply(is_math_question)
flagged_indices = test_df[math_mask].index.tolist()
print(f"Flagged {len(flagged_indices)} / {len(test_df)} rows as math-type")

overridden = 0
override_records = []
BATCH_SIZE = CONFIG["BATCH_SIZE"]
n_batches = (len(flagged_indices) + BATCH_SIZE - 1) // BATCH_SIZE

pbar = tqdm(range(0, len(flagged_indices), BATCH_SIZE), total=n_batches, desc="Verifying math rows (batched)")
for batch_num, batch_start in enumerate(pbar, start=1):
    batch_idx = flagged_indices[batch_start: batch_start + BATCH_SIZE]
    batch_rows = test_df.loc[batch_idx]
    questions = batch_rows["question"].tolist()
    answers = batch_rows["answer"].tolist()

    batch_results = solve_and_verify_batch(questions, answers)

    for j, idx in enumerate(batch_idx):
        row = test_df.loc[idx]
        is_faithful, computed_ans, expr, given_numeric = batch_results[j]
        if is_faithful is not None:
            is_hallucinated = not is_faithful
            new_label = int(is_hallucinated) if CONFIG["HALLUCINATED_LABEL_VALUE"] == 1 else int(not is_hallucinated)
            submission.loc[submission["id"] == row["id"], "label"] = new_label
            overridden += 1
            override_records.append({
                "id": row["id"],
                "question": row["question"],
                "answer": row["answer"],
                "expr": expr,
                "computed_value": computed_ans,
                "given_numeric_parsed": given_numeric,
                "final_label": new_label,
            })

    pbar.set_postfix(overridden=overridden, rows=min(batch_start + BATCH_SIZE, len(flagged_indices)))

    if batch_num % CONFIG["CHECKPOINT_EVERY"] == 0:
        submission.to_csv(CONFIG["SUBMISSION_PATH"], index=False)
        if override_records:
            pd.DataFrame(override_records).to_csv(CONFIG["OVERRIDE_LOG_PATH"], index=False)
        tqdm.write(f"Checkpoint saved at batch {batch_num}/{n_batches} "
                   f"({min(batch_start + BATCH_SIZE, len(flagged_indices))}/{len(flagged_indices)} rows). "
                   f"Overridden so far: {overridden}")

print(f"\nOverrode {overridden} / {len(flagged_indices)} flagged rows.")
assert submission["label"].isin([0, 1]).all(), "labels must be 0/1"
submission.to_csv(CONFIG["SUBMISSION_PATH"], index=False)

if override_records:
    pd.DataFrame(override_records).to_csv(CONFIG["OVERRIDE_LOG_PATH"], index=False)
    print(f"Saved override log for manual inspection to: {CONFIG['OVERRIDE_LOG_PATH']}")

print(f"Final save completed successfully to: {CONFIG['SUBMISSION_PATH']}")

Writing math_verifier_qwen_v5.py


In [15]:
!python math_verifier_qwen_v5.py \
    --test_csv_path "/kaggle/input/competitions/bengali-hallucination/test set.csv" \
    --submission_path "/kaggle/working/submission.csv" \
    --model_name "unsloth/Qwen3-14B" \
    --batch_size 8 \
    --checkpoint_every 5

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading unsloth/Qwen3-14B on single GPU for math verification...
==((====))==  Unsloth 2026.7.3: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Loading weights: 100%|████████████████████████| 443/443 [01:05<00:00,  6.82it/s]
Flagged 264 / 2516 rows as math-type
Verifying math rows (batched):   0%|                     | 0/33 [00:00<?, ?it/s]Both `max_new_tokens` (=768) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation

In [16]:
import pandas as pd

sub = pd.read_csv("/kaggle/working/submission.csv")
test_df = pd.read_parquet("/kaggle/working/test_augmented.parquet")

merged = test_df.merge(sub, on="id", how="inner")
assert len(merged) == len(sub) == len(test_df), "row count mismatch after merge — check id overlap"

sample = merged.sample(n=50, random_state=3000)

for _, row in sample.iterrows():
    print(f"id: {row['id']}")
    print(f"context_augmented:\n{row['context_augmented']}")
    print(f"question: {row['question']}")
    print(f"answer: {row['answer']}")
    print(f"label: {row['label']}  ({'faithful' if row['label'] == 1 else 'hallucinated'})")
    print("-" * 80)

id: 1912
context_augmented:
ক্লিক অ্যাসিড আরএনএ পলিমারেজ রুবিসকো রিবোসোমাল আরএনএ ল ল্যাকটেজ ল্যাকটিক অ্যাসিড ল্যাকটোজ ল্যানোলিন লাউরিক এসিড লেকটিন লেপটিন লেপটোমাইসিন বি লিউসিন লিউকোট্রিন লিগাসে লিগনিন লিমোনিন লিনালুল লিনোলিক অ্যাসিড লিনোলিক অ্যাসিড লিপেজ লিপিড লিপিড নোঙ্গরযুক্ত প্রোটিন লিপোমাইড লিপোপ্রোটিন কম ঘনত্বের লিপোপ্রোটিন, এলডিএল লুটিনাইজিং হরমোন (এলএইচ) লাইকোপেন লাইসিন লাইসোজাইম শ ষ স সাফ্রল স্যালিসিলডিহাইড স্যালিসিলিক অ্যাসিড সালভিনোরিন-এ – সি 23 এইচ 28 ও 8 স্যাপোনিন সিক্রেটিন সেলেনোসিস্টাইন সেলেনোমেথিওনিন সেলেনোপ্রোটিন সেরিন সেরিন কিনেস সেরোটোনিন স্কটোল সংকেত স্বীকৃতি কণা সোমাটোস্ট্যাটিন সরবিক এসিড সেলুলেজ সেলুলোজ - (C 6 H 10 O 5 ) x সেরুলেনিন সেট্রিমোনিয়াম ব্রোমাইড (Cetrimide) - C 19 H 42 BrN স্ফিংগোলিপিড সিনামালডিহাইড সিট্রাল সাইট্রিক অ্যাসিড সিট্রিনিন α-সাইক্লোডেক্সট্রিন সাইক্লোডেক্সট্রিন গ্লাইকোসিলট্রান্সফেরেজ সাইক্লোঅক্সিজেনেস সাইক্লোপামিন সাইক্লোপিয়াজোনিক অ্যাসিড সিস্টাইন সিস্টাইন সেরোটোনিন সাইটিডিন সাইটোক্যালাসিন সাইটোকালাসিন ই সাইটোক্রোম সাইটোক্রোম সি সাইটোক্রোম সি 